##TASK1:Custom Document Loader

In [1]:
!pip install -q sentence-transformers
!pip install -q langchain-text-splitters
!pip install -q scikit-learn

In [39]:
!pip install streamlit
!pip install cohere
!pip install sentence-transformers
!pip install langchain-text-splitters
!pip install scikit-learn
!pip install numpy
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸
changed 22 packages in 1s
⠼
⠼3 packages are looking for funding
⠼  run `npm fund` for details
⠼

In [2]:
# Text splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embedding model
from sentence_transformers import SentenceTransformer

# Similarity calculation
from sklearn.metrics.pairwise import cosine_similarity

# Numerical operations
import numpy as np

# File handling
import os

In [3]:
# Sample text for testing

text = """
Artificial Intelligence (AI) is a branch of computer science that enables machines
to perform tasks that normally require human intelligence.

Machine Learning is a subset of Artificial Intelligence that allows computers
to learn from data without being explicitly programmed.

Deep Learning is a subset of Machine Learning that uses artificial neural networks.

Natural Language Processing (NLP) enables computers to understand and generate
human language.

Computer Vision enables machines to identify and understand images and videos.

Artificial Intelligence is widely used in healthcare, banking, education,
transportation, robotics, cybersecurity, and many other industries.
"""

# Save the text into a TXT file

with open("sample.txt", "w", encoding="utf-8") as file:
    file.write(text)

print("Text file created successfully!")

Text file created successfully!


In [4]:
with open("sample.txt", "r", encoding="utf-8") as file:
    data = file.read()

print(data)


Artificial Intelligence (AI) is a branch of computer science that enables machines
to perform tasks that normally require human intelligence.

Machine Learning is a subset of Artificial Intelligence that allows computers
to learn from data without being explicitly programmed.

Deep Learning is a subset of Machine Learning that uses artificial neural networks.

Natural Language Processing (NLP) enables computers to understand and generate
human language.

Computer Vision enables machines to identify and understand images and videos.

Artificial Intelligence is widely used in healthcare, banking, education,
transportation, robotics, cybersecurity, and many other industries.



In [5]:
def load_and_chunk_document(file_path, chunk_size=300, overlap=50):
    """
    Reads a text document, splits it into chunks,
    and returns a list of text chunks.

    Parameters:
        file_path (str): Path of the text file.
        chunk_size (int): Maximum number of characters in one chunk.
        overlap (int): Number of overlapping characters.

    Returns:
        list: List of text chunks.
    """

    # Step 1: Check whether the file exists
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"{file_path} does not exist.")

    # Step 2: Open the text file
    with open(file_path, "r", encoding="utf-8") as file:
        document = file.read()

    # Step 3: Create a text splitter
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        separators=["\n\n", "\n", ".", " ", ""]
    )

    # Step 4: Split the document
    chunks = splitter.split_text(document)

    # Step 5: Remove empty chunks
    chunks = [chunk.strip() for chunk in chunks if chunk.strip()]

    # Step 6: Return the chunks
    return chunks

In [6]:
chunks = load_and_chunk_document("sample.txt")

In [7]:
print("Total Chunks:", len(chunks))

Total Chunks: 3


Displaying All Chunks

In [8]:
for i, chunk in enumerate(chunks, start=1):
    print(f"\nChunk {i}")
    print("-" * 50)
    print(chunk)


Chunk 1
--------------------------------------------------
Artificial Intelligence (AI) is a branch of computer science that enables machines
to perform tasks that normally require human intelligence.

Machine Learning is a subset of Artificial Intelligence that allows computers
to learn from data without being explicitly programmed.

Chunk 2
--------------------------------------------------
Deep Learning is a subset of Machine Learning that uses artificial neural networks.

Natural Language Processing (NLP) enables computers to understand and generate
human language.

Computer Vision enables machines to identify and understand images and videos.

Chunk 3
--------------------------------------------------
Artificial Intelligence is widely used in healthcare, banking, education,
transportation, robotics, cybersecurity, and many other industries.


##TASK 2:Custom Embedding Function

Create Embeddings

In [9]:
from sentence_transformers import SentenceTransformer

def create_embeddings(chunks, model_name="all-MiniLM-L6-v2"):
    """
    Creates embeddings for all text chunks.
    """

    # Load Sentence Transformer model
    model = SentenceTransformer(model_name)

    # Generate embeddings
    embeddings = model.encode(chunks)

    # Convert NumPy array to Python list
    embeddings = embeddings.tolist()

    return embeddings

In [10]:
embeddings = create_embeddings(chunks)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [11]:
print("Number of embeddings:", len(embeddings))
print("Dimension of first embedding:", len(embeddings[0]))

Number of embeddings: 3
Dimension of first embedding: 384


##TASK 3: Search Function (Programming)

In [12]:
def search_chunks(query, chunks, embeddings, k=3):
    """
    Searches for the top-k most similar chunks.
    """

    # Load the same embedding model
    model = SentenceTransformer("all-MiniLM-L6-v2")

    # Create embedding for the query
    query_embedding = model.encode([query])

    # Calculate cosine similarity
    similarity_scores = cosine_similarity(query_embedding, embeddings)[0]

    # Get indices of top-k chunks
    top_indices = np.argsort(similarity_scores)[::-1][:k]

    # Store top chunks
    top_chunks = []

    for index in top_indices:
        top_chunks.append(chunks[index])

    return top_chunks

Testing The Function

In [13]:
query = "What is Artificial Intelligence?"

results = search_chunks(query, chunks, embeddings)

print("Top Matching Chunks:\n")

for i, chunk in enumerate(results, start=1):
    print(f"Result {i}")
    print("-" * 50)
    print(chunk)
    print()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Top Matching Chunks:

Result 1
--------------------------------------------------
Artificial Intelligence (AI) is a branch of computer science that enables machines
to perform tasks that normally require human intelligence.

Machine Learning is a subset of Artificial Intelligence that allows computers
to learn from data without being explicitly programmed.

Result 2
--------------------------------------------------
Artificial Intelligence is widely used in healthcare, banking, education,
transportation, robotics, cybersecurity, and many other industries.

Result 3
--------------------------------------------------
Deep Learning is a subset of Machine Learning that uses artificial neural networks.

Natural Language Processing (NLP) enables computers to understand and generate
human language.

Computer Vision enables machines to identify and understand images and videos.



##Before doing Task 4 we will do Task 5

##TASK 5: Add Cohere Answer Generation (Programming)

In [14]:
!pip install -q cohere

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 357.0/357.0 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 77.8 MB/s eta 0:00:00


In [15]:
import cohere

In [16]:
api_key = input("Enter your Cohere API Key: ")

Enter your Cohere API Key: IfCCFT9tuHPrCafFiiH60tcNsxI4Soqn2BUIoB4x


In [17]:
import cohere

def generate_answer(query, context, api_key):
    """
    Generate an answer using Cohere based on the retrieved context.

    Parameters:
        query (str): User question
        context (str): Retrieved document chunks
        api_key (str): Cohere API key

    Returns:
        str: Generated answer
    """

    # Initialize Cohere client
    co = cohere.Client(api_key)

    # Create prompt
    prompt = f"""
You are a helpful AI assistant.

Answer the user's question ONLY using the information provided below.

Context:
{context}

Question:
{query}

Answer:
"""

    # Generate response
    response = co.chat(
        model="command-a-03-2025",
        message=prompt
    )

    return response.text

Testing the Function

In [18]:
query = "What is Artificial Intelligence?"

results = search_chunks(query, chunks, embeddings)

context = ""

for result in results:
    if isinstance(result, dict):
        context += result["Chunk"] + "\n\n"
    else:
        context += result + "\n\n"

answer = generate_answer(query, context, api_key)

print(answer)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Artificial Intelligence (AI) is a branch of computer science that enables machines to perform tasks that normally require human intelligence.


##TASK 4:Build a Simple Streamlit App (Programming)

In [19]:
!pip install -q streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 41.7 MB/s eta 0:00:00


In [20]:
%%writefile app.py

import streamlit as st
import os
import numpy as np
import cohere

from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

Writing app.py


ADDING TASK 1 FUNCTION

In [21]:
%%writefile -a app.py

def load_and_chunk_document(file_path, chunk_size=300, overlap=50):

    with open(file_path, "r", encoding="utf-8") as file:
        document = file.read()

    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap
    )

    chunks = splitter.split_text(document)

    return chunks

Appending to app.py


ADDING TASK 2 FUNCTION

In [22]:
%%writefile -a app.py

def create_embeddings(chunks):

    model = SentenceTransformer("all-MiniLM-L6-v2")

    embeddings = model.encode(chunks)

    return embeddings.tolist()

Appending to app.py


ADDING TASK 3 FUNCTION

In [23]:
%%writefile -a app.py

def search_chunks(query, chunks, embeddings, k=3):

    model = SentenceTransformer("all-MiniLM-L6-v2")

    query_embedding = model.encode([query])

    scores = cosine_similarity(query_embedding, embeddings)[0]

    top_indices = np.argsort(scores)[::-1][:k]

    results = []

    for idx in top_indices:
        results.append(chunks[idx])

    return results

Appending to app.py


ADDING TASK 5 FUNCTION

In [24]:
%%writefile -a app.py

def generate_answer(query, context, api_key):

    co = cohere.Client(api_key)

    prompt = f"""
You are a helpful assistant.

Context:
{context}

Question:
{query}

Answer:
"""

    response = co.chat(
        model="command-a-03-2025",
        message=prompt
    )

    return response.text

Appending to app.py


STREAMLIT INTERFACE

In [25]:
%%writefile -a app.py

st.title("📚 Simple RAG Application")

uploaded_file = st.file_uploader("Upload a TXT file", type=["txt"])

query = st.text_input("Enter your question")

api_key = st.text_input("Enter Cohere API Key", type="password")

if st.button("Search"):

    if uploaded_file is not None and query and api_key:

        with open("uploaded.txt", "wb") as f:
            f.write(uploaded_file.getbuffer())

        chunks = load_and_chunk_document("uploaded.txt")

        embeddings = create_embeddings(chunks)

        results = search_chunks(query, chunks, embeddings)

        st.subheader("Top Matching Chunks")

        context = ""

        for chunk in results:
            st.write(chunk)
            context += chunk + "\n\n"

        answer = generate_answer(query, context, api_key)

        st.subheader("Generated Answer")

        st.success(answer)

    else:
        st.warning("Please upload a file, enter a question, and provide your API key.")

Appending to app.py


In [33]:
!ls

app.py	sample_data  sample.txt


In [40]:
!nohup streamlit run app.py > log.txt 2>&1 &

In [41]:
!cat log.txt



2026-07-23 13:59:47.584 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.118.240.109:8501



In [ ]:
!streamlit run app.py & npx localtunnel --port 8501

⠙⠹

⠸⠼⠴⠦2026-07-23 14:05:07.035 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.118.240.109:8501

your url is: https://easy-mammals-exist.loca.lt
Loading weights: 100% 103/103 [00:00<00:00, 6559.27it/s]
Loading weights: 100% 103/103 [00:00<00:00, 14459.24it/s]
Loading weights: 100% 103/103 [00:00<00:00, 6464.65it/s]
Loading weights: 100% 103/103 [00:00<00:00, 7052.01it/s]
